# ViT-Up inference

Query dense features of an image and compare them with the backbone features via PCA.

In [ ]:
import matplotlib.pyplot as plt
import torch
from PIL import Image

from vit_up.inference import ViTUpWrapper
from vit_up.visualization import PCA

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Releases: dinov3_splus, dinov3_splus_stage1, dinov3_splus_g28_56, dinov3_base, dinov3_base_stage1, dinov2_small
model = ViTUpWrapper("dinov3_splus", device=device, use_bfloat16=device == "cuda", query_chunk_size=16384)

In [ ]:
IMAGE = "../assets/fruit_store.png"

image = Image.open(IMAGE).convert("RGB")
side = min(image.size)
image = image.crop(((image.width - side) // 2, (image.height - side) // 2,
                    (image.width + side) // 2, (image.height + side) // 2))

features = model.dense(image, output_size=448)[0].cpu()   # (448, 448, C), dense ViT-Up features
backbone = model.backbone_features()[0].cpu()             # (28, 28, C), backbone features at 448 px
print(features.shape, backbone.shape)

In [ ]:
pca = PCA(backbone)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, shown) in zip(axes, [("Image", image), ("Backbone", pca.rgb(backbone)), ("ViT-Up", pca.rgb(features))]):
    ax.imshow(shown, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Features at arbitrary continuous (x, y) coordinates in [0, 1], here 5 random points.
queries = torch.rand(1, 5, 2)
point_features = model(query_coords=queries)
print(point_features.shape)  # (1, 5, C)